# 17 — Multipartite Entanglement from Frequency-Mode Graphs

**Engineering question**

How do many squeezed frequency pairs become a scalable quantum network?

Notebook 13 established two-mode squeezed pair resources.  
Notebook 17 turns those pair resources into a graph view:

```text
two-mode squeezed pairs
→ frequency modes as nodes
→ quantum correlations as edges
→ multipartite graph
→ cluster-state resource
```

Engineering statement:

> Coupling many squeezed frequency pairs transforms independent quantum resources into a multipartite entangled frequency graph.


## Setup

This notebook creates:

```text
figures/
results/csv/
results/json/
results/17_outputs.zip
```

The final cell supports both:

- **Google Colab**: starts a real browser download with `files.download(...)`
- **Jupyter**: displays a clickable `FileLink`


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Pair resources versus graph resources

Notebook 13 treated symmetric mode pairs as the primitive resource.

That is useful, but it is still pairwise:

```text
ω₋₃ ↔ ω₊₃
ω₋₂ ↔ ω₊₂
ω₋₁ ↔ ω₊₁
```

Multipartite entanglement asks for a richer object:

```text
nodes = frequency modes
edges = quantum correlations
```

The system becomes a graph rather than a list of independent pairs.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# Left: independent pairs
ax = axes[0]
ax.set_title("Independent pair resources", fontsize=14)
ax.axis("off")
pairs = [(-3, 3), (-2, 2), (-1, 1)]
ys = [2.5, 1.6, 0.7]

for (a, b), y in zip(pairs, ys):
    ax.plot([0.1, 0.9], [y, y], linewidth=2.5)
    ax.scatter([0.1, 0.9], [y, y], s=120)
    ax.text(0.02, y, fr"$\omega_{{{a}}}$", ha="right", va="center", fontsize=12)
    ax.text(0.98, y, fr"$\omega_{{+{b}}}$", ha="left", va="center", fontsize=12)

ax.text(0.5, 0.05, "separate EPR-like pairs", ha="center", fontsize=11)
ax.set_xlim(-0.15, 1.15)
ax.set_ylim(0, 3.0)

# Right: graph entanglement
ax = axes[1]
ax.set_title("Multipartite frequency graph", fontsize=14)
ax.axis("off")

nodes = np.arange(-3, 4)
x = np.linspace(0.05, 0.95, len(nodes))
y = np.ones_like(x) * 1.55

# chain edges
for i in range(len(x) - 1):
    ax.plot([x[i], x[i + 1]], [y[i], y[i + 1]], linewidth=2)

# cross couplings
cross_edges = [(0, 3), (1, 4), (2, 5), (0, 6), (1, 5)]
for i, j in cross_edges:
    ax.plot([x[i], x[j]], [y[i], y[j]], linewidth=1.6, alpha=0.65)

ax.scatter(x, y, s=140, zorder=3)
for xi, ni in zip(x, nodes):
    label = r"$\omega_0$" if ni == 0 else fr"$\omega_{{{ni:+d}}}$"
    ax.text(xi, 1.18, label, ha="center", fontsize=10)

ax.text(0.5, 0.05, "modes become connected network nodes", ha="center", fontsize=11)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0, 2.7)

fig.suptitle("Independent Pairs Become an Entangled Frequency Graph", fontsize=16)

savefig(fig, "17_pairs_to_graph.png")
plt.show()


## 2. Multipartite adjacency matrix

Notebook 13 used a pair-correlation matrix: mostly symmetric entries.

A multipartite graph can include:

- nearest-neighbor frequency couplings,
- symmetric pair couplings,
- cross-couplings,
- pump-centered structure.

This matrix is a specification diagram, not measured covariance data.


In [ ]:
modes = np.arange(-5, 6)
N = len(modes)
A = np.zeros((N, N))

# nearest-neighbor chain
for i in range(N - 1):
    A[i, i + 1] = A[i + 1, i] = 0.45

# symmetric pair edges
for i, a in enumerate(modes):
    for j, b in enumerate(modes):
        if a + b == 0 and a != 0:
            A[i, j] = max(A[i, j], 0.9)

# cross-couplings
extra = [(-5, 1), (-4, 2), (-3, 1), (-2, 4), (-1, 3), (1, 5)]
idx = {m: k for k, m in enumerate(modes)}
for a, b in extra:
    A[idx[a], idx[b]] = A[idx[b], idx[a]] = 0.65

np.fill_diagonal(A, 0)

fig, ax = plt.subplots(figsize=(7.2, 6.4))
im = ax.imshow(A, vmin=0, vmax=1)

ax.set_title("Multipartite Correlations Extend Beyond Pairwise Squeezing", fontsize=14)
ax.set_xticks(np.arange(N))
ax.set_yticks(np.arange(N))
ax.set_xticklabels([f"{m:+d}" if m != 0 else "0" for m in modes])
ax.set_yticklabels([f"{m:+d}" if m != 0 else "0" for m in modes])
ax.set_xlabel("Frequency mode")
ax.set_ylabel("Frequency mode")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="edge weight")

savefig(fig, "17_adjacency_matrix.png")
plt.show()


## 3. Frequency comb as an entanglement graph

A graph representation lets the notebook stop thinking only in spectra.

Each frequency mode is a node.  
Each quantum correlation is an edge.

The same comb can now be described as:

```text
frequency resource
→ graph resource
→ cluster-state candidate
```


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.axis("off")

# two rows of frequency modes
top_modes = np.arange(-5, 1)
bottom_modes = np.arange(1, 7)
x_top = np.linspace(0.08, 0.92, len(top_modes))
x_bot = np.linspace(0.08, 0.92, len(bottom_modes))
y_top = np.ones_like(x_top) * 0.7
y_bot = np.ones_like(x_bot) * 0.25

# horizontal edges
for xs, ys in [(x_top, y_top), (x_bot, y_bot)]:
    for i in range(len(xs) - 1):
        ax.plot([xs[i], xs[i + 1]], [ys[i], ys[i + 1]], linewidth=2)

# vertical/diagonal graph edges
for i in range(len(x_top)):
    ax.plot([x_top[i], x_bot[i]], [y_top[i], y_bot[i]], linewidth=1.5, alpha=0.7)
for i in range(len(x_top) - 1):
    ax.plot([x_top[i], x_bot[i + 1]], [y_top[i], y_bot[i + 1]], linewidth=1.2, alpha=0.55)
    ax.plot([x_bot[i], x_top[i + 1]], [y_bot[i], y_top[i + 1]], linewidth=1.2, alpha=0.55)

ax.scatter(x_top, y_top, s=135, zorder=3)
ax.scatter(x_bot, y_bot, s=135, zorder=3)

for xi, mode in zip(x_top, top_modes):
    label = r"$\omega_0$" if mode == 0 else fr"$\omega_{{{mode:+d}}}$"
    ax.text(xi, y_top[0] + 0.12, label, ha="center", fontsize=10)

for xi, mode in zip(x_bot, bottom_modes):
    ax.text(xi, y_bot[0] - 0.15, fr"$\omega_{{+{mode}}}$", ha="center", fontsize=10)

ax.text(0.5, 0.03, "nodes = frequency modes; edges = quantum correlations", ha="center", fontsize=12)
ax.set_title("Frequency Comb as an Entanglement Graph", fontsize=16)
ax.set_xlim(0, 1)
ax.set_ylim(-0.08, 0.95)

savefig(fig, "17_frequency_graph.png")
plt.show()


## 4. Graph scaling

As the number of usable modes increases, the graph can support more nodes and edges.

This figure is intentionally simple:

- node count grows with addressable modes,
- edge count grows faster when cross-couplings are available,
- usable cluster-state size is ultimately limited by loss, squeezing, and measurement quality.


In [ ]:
nodes = np.arange(4, 81, 4)
chain_edges = nodes - 1
cross_edges = np.round(1.6 * nodes).astype(int)
possible_edges = chain_edges + cross_edges

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(nodes, chain_edges, marker="o", linewidth=2.2, label="nearest-neighbor edges")
ax.plot(nodes, possible_edges, marker="s", linewidth=2.2, label="with cross-couplings")

ax.set_title("Larger Frequency Combs Support Larger Quantum Graphs", fontsize=16)
ax.set_xlabel("Usable frequency modes / graph nodes")
ax.set_ylabel("Graph edges / correlation links")
ax.grid(True, alpha=0.3)
ax.legend()

ax.text(34, 26, "actual usable size is\nloss- and squeezing-limited", fontsize=10)

savefig(fig, "17_graph_scaling.png")
plt.show()


## 5. Resource pipeline

The repository has now built the pipeline:

```text
scaling pressure
→ frequency multiplexing
→ frequency comb
→ Kerr mixing
→ two-mode squeezing
→ multipartite graph
→ cluster-state resource
```

This figure connects the notebooks rather than introducing new physics.


In [ ]:
labels = [
    "Pump",
    "χ³ Kerr",
    "Four-wave\nmixing",
    "Two-mode\nsqueezing",
    "Frequency-bin\npairs",
    "Graph\nentanglement",
    "Cluster-state\nresource",
]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(13, 3.2))

for xi, label in zip(x, labels):
    rect = plt.Rectangle((xi - 0.42, -0.25), 0.84, 0.5, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(xi, 0, label, ha="center", va="center", fontsize=10)

for i in range(len(labels) - 1):
    ax.annotate("", xy=(i + 0.58, 0), xytext=(i + 0.42, 0), arrowprops=dict(arrowstyle="->", linewidth=2))

ax.set_title("Resource Pipeline: From Kerr Mixing to Cluster-State Graphs", fontsize=16)
ax.set_xlim(-0.7, len(labels) - 0.3)
ax.set_ylim(-0.65, 0.7)
ax.axis("off")

savefig(fig, "17_resource_pipeline.png")
plt.show()


## 6. Engineering summary

The notebook's engineering shift is:

```text
frequency pair
→ graph node pair
→ graph network
→ cluster-state candidate
```

This prepares later notebooks on measurement, loss, and fault-tolerant architectures.


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Object": "Optical channel",
            "Graph interpretation": "Frequency-bin node",
            "Engineering role": "Addressable quantum mode",
            "Next constraint": "Mode control",
        },
        {
            "Object": "Two-mode squeezed pair",
            "Graph interpretation": "Weighted edge",
            "Engineering role": "Pairwise entanglement primitive",
            "Next constraint": "Edge strength",
        },
        {
            "Object": "Frequency comb",
            "Graph interpretation": "Node set",
            "Engineering role": "Scalable graph substrate",
            "Next constraint": "Loss and bandwidth",
        },
        {
            "Object": "Mode coupling",
            "Graph interpretation": "Cross edge",
            "Engineering role": "Multipartite connectivity",
            "Next constraint": "Programmability",
        },
        {
            "Object": "Cluster state",
            "Graph interpretation": "Computational graph",
            "Engineering role": "Measurement-based resource",
            "Next constraint": "Fault tolerance",
        },
    ]
)

fig, ax = plt.subplots(figsize=(12.5, 3.6))
ax.axis("off")

table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(9.6)
table.scale(1.1, 1.65)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Engineering Summary: Frequency Modes Become Quantum Network Nodes", fontsize=16, pad=20)

savefig(fig, "17_summary.png")

summary.to_csv(CSV / "17_summary.csv", index=False)
summary.to_json(JS / "17_summary.json", orient="records", indent=2)

plt.show()
summary


## 7. Export and download

This cell packages all outputs and starts a download.

It uses Colab's native downloader when available.  
For local Jupyter, it falls back to a clickable `FileLink`.


In [ ]:
zip_path = RES / "17_outputs.zip"

files_to_zip = [
    FIG / "17_pairs_to_graph.png",
    FIG / "17_adjacency_matrix.png",
    FIG / "17_frequency_graph.png",
    FIG / "17_graph_scaling.png",
    FIG / "17_resource_pipeline.png",
    FIG / "17_summary.png",
    CSV / "17_summary.csv",
    JS / "17_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Takeaways

- Two-mode squeezing gives pairwise entanglement primitives.
- Multipartite entanglement appears when frequency modes are treated as a graph.
- The frequency comb provides the node set.
- Quantum correlations provide the edge set.
- Cluster states are graph resources whose usefulness depends on squeezing, loss, measurement, and feedforward.

## Next notebook

**23 — Cluster-State Architecture**

Notebook 17 builds frequency-mode graphs.

Notebook 23 should ask how those graphs become usable continuous-variable cluster-state resources:

```text
frequency graph
→ cluster-state geometry
→ measurement pattern
→ feedforward
→ computation / networking resource
```
